In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests
from tqdm import tqdm
from IPython.display import clear_output

from numpy.linalg import svd
from scipy.sparse.linalg import svds
from numpy.lib.stride_tricks import as_strided
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller, acf, pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
import hampel

import warnings
warnings.simplefilter(action='ignore', category=Warning)


In [ ]:
df = pd.read_csv('data/open_meteo_train.csv')
df['time'] = pd.to_datetime(df['time'])
df['month'] = df['time'].dt.month
# df['DD'] = df['DD'] % 360

In [ ]:
df

# Автокорреляция

In [ ]:
for site, group in df.groupby('site'):
    plot_acf(group['pm25'], alpha=0.05, lags=24*3)
    plt.title(site)
    plt.show()

In [ ]:
# Параметры анализа
lags = 24*2  # Исследуем двое суток (48 часов)
all_acfs = []
sites = []

plt.figure(figsize=(10, 7))

# Проходим циклом по датчикам
for site, group in df.groupby('site'):
    # Важно: работаем с колонкой pm25, удаляя пропуски для расчета ACF
    y = group['pm25'].dropna()
    
    if len(y) > lags:
        corel = pacf(y, nlags=lags)
        all_acfs.append(corel)
        sites.append(site)
        
        # Рисуем тонкую серую линию для каждого датчика
        plt.plot(corel, color='gray', alpha=0.2, linewidth=1)

# Вычисляем средний профиль автокорреляции по всей сети
mean_acf = np.mean(all_acfs, axis=0)

# Отрисовка жирной средней линии
plt.plot(mean_acf, '-o', color='#1f77b4', linewidth=1.5, label='Средняя PACF по сети')

# Добавляем вертикальную линию на 24 часа для акцента
# plt.axvline(x=24, color='red', linestyle='--', alpha=0.6, label='Суточный цикл (24ч)')

# Оформление графика

plt.xlabel('Временной лаг (часы)', fontsize=18)
plt.ylabel('Коэффициент частной автокорреляции', fontsize=18)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(fontsize=18)

plt.tight_layout()
plt.show()

In [ ]:
# Параметры анализа
lags = 24*2  # Исследуем двое суток (48 часов)
all_acfs = []
sites = []

plt.figure(figsize=(10, 6))

# Проходим циклом по датчикам
for site, group in df.groupby('site'):
    # Важно: работаем с колонкой pm25, удаляя пропуски для расчета ACF
    y = group['pm25'].dropna()
    
    if len(y) > lags:
        corel = acf(y, nlags=lags)
        all_acfs.append(corel)
        sites.append(site)
        
        # Рисуем тонкую серую линию для каждого датчика
        plt.plot(corel, color='gray', alpha=0.2, linewidth=1)

# Вычисляем средний профиль автокорреляции по всей сети
mean_acf = np.mean(all_acfs, axis=0)

# Отрисовка жирной средней линии
plt.plot(mean_acf, color='#1f77b4', linewidth=2.5, label='Средняя ACF по сети')

# Добавляем вертикальную линию на 24 часа для акцента
plt.axvline(x=24, color='red', linestyle='--', alpha=0.6, label='Суточный цикл (24ч)')

# Оформление графика
plt.xlabel('Временной лаг (часы)', fontsize=18)
plt.ylabel('Коэффициент автокорреляции', fontsize=18)
plt.grid(True, linestyle=':', alpha=0.7)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.legend(fontsize=18)

plt.tight_layout()
plt.show()

# Удаление шумов фильтром Хампеля

In [ ]:
k_range = [*range(1,13)]
site_metrics = {}
for site, group in df.groupby('site'):
    MSE = []
    MAE = []
    for k in k_range:
        clear_output(wait=True)
        print(site, k)
        filtered_data = hampel.hampel(group['pm25'],
                                      window_size=k).filtered_data.values
        MSE.append(mean_squared_error(group['pm25'].values,
                                      filtered_data))
        MAE.append(mean_absolute_error(group['pm25'].values,
                                      filtered_data))
    site_metrics[site] = {'MSE':MSE,'MAE':MAE}

In [ ]:
plt.figure(figsize=(5,5))
mean_MSE = []
for i , site in enumerate(site_metrics.keys()):
    mean_MSE.append(site_metrics[site]['MSE'])
    plt.plot(k_range,site_metrics[site]['MSE'], color='black',alpha=0.2)
plt.plot(k_range, np.mean(np.array(mean_MSE),axis=0), label='mean',color='blue')
plt.plot(k_range, np.median(np.array(mean_MSE),axis=0), label='median',color='red')
plt.xlabel('k')
plt.title('MSE')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(5,5))
mean_MAE = []
for i , site in enumerate(site_metrics.keys()):
    mean_MAE.append(site_metrics[site]['MAE'])
    plt.plot(k_range,site_metrics[site]['MAE'], color='black',alpha=0.2)
plt.plot(k_range, np.mean(np.array(mean_MAE),axis=0), label='mean',color='blue')
plt.plot(k_range, np.median(np.array(mean_MAE),axis=0), label='median',color='red')
plt.legend()
plt.xlabel('k')
plt.title('MAE')
plt.grid()
plt.show()

In [ ]:
plt.plot(df[df['site']==3837]['pm25'],label='orig')
plt.plot(hampel.hampel(df[df['site']==3837]['pm25'],window_size=7).filtered_data.values,label='filtered',alpha=0.5)
plt.show()

# Разложение с seasonal_decompose

In [ ]:
for site, group in df.groupby('site'):
    group['time'] = pd.to_datetime(group['time'])
    # group['pm25'] = hampel.hampel(group['pm25'],window_size=7).filtered_data.values
    decompose = seasonal_decompose(group.set_index('time')['pm25'],period=24)
    print(site)
    plt.figure(figsize=(8,6))
    plt.subplot(3,1,1)
    plt.plot(decompose.trend.iloc[:])
    # plt.title('Trend')
    plt.xticks([])

    plt.subplot(3,1,2)
    plt.plot(decompose.seasonal.iloc[:200])
    # plt.title('Seasonal')
    plt.xticks(fontsize=8)
    # plt.xticks(decompose.seasonal.index[:200])

    plt.subplot(3,1,3)
    plt.plot(decompose.resid.iloc[:200])
    # plt.title('Residuals')
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
for site, group in df.groupby('site'):
    group['time'] = pd.to_datetime(group['time'])
    date_index = pd.date_range("11.01.2021","02.01.2022",freq='1h')
    decompose = seasonal_decompose(group.set_index('time')['pm25'].loc[date_index],period=24)
    print(site)
    plt.figure(figsize=(8,6))
    plt.subplot(3,1,1)
    plt.plot(decompose.trend.iloc[:])
    # plt.title('Trend')
    plt.xticks([])

    plt.subplot(3,1,2)
    plt.plot(decompose.seasonal.iloc[:200])
    # plt.title('Seasonal')
    plt.xticks(fontsize=8)
    # plt.xticks(decompose.seasonal.index[:200])

    plt.subplot(3,1,3)
    plt.plot(decompose.resid.iloc[:200])
    # plt.title('Residuals')
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
for site, group in df.groupby('site'):
    group['time'] = pd.to_datetime(group['time'])
    date_index = pd.date_range("06.01.2021","10.01.2022",freq='1h')
    decompose = seasonal_decompose(group.set_index('time')['pm25'].loc[date_index],period=24)
    print(site)
    plt.figure(figsize=(8,6))
    plt.subplot(3,1,1)
    plt.plot(decompose.trend.iloc[:])
    # plt.title('Trend')
    plt.xticks([])

    plt.subplot(3,1,2)
    plt.plot(decompose.seasonal.iloc[:200])
    # plt.title('Seasonal')
    plt.xticks(fontsize=8)
    # plt.xticks(decompose.seasonal.index[:200])

    plt.subplot(3,1,3)
    plt.plot(decompose.resid.iloc[:200])
    # plt.title('Residuals')
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
s = df[df['site'] == 3857]
s['time'] = pd.to_datetime(s['time'])
decompose = seasonal_decompose(s.set_index('time')['pm25'],period=24)

plt.figure(figsize=(14,3))
plt.plot(decompose.trend.iloc[:])
plt.grid()
plt.title('Тренд')
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(12,3))
plt.plot(decompose.seasonal.iloc[:24*7])
ax = plt.gca()
# Настройка Major Grid (каждые 24 часа)
ax.xaxis.set_major_locator(mdates.HourLocator(byhour=0))
# plt.grid(which='major', color='gray', linestyle='-', alpha=0.5)
# # Настройка Minor Grid (каждые 6 часов)
ax.xaxis.set_major_locator(mdates.HourLocator(byhour=[8, 0]))
plt.grid(which='major', color='gray', linestyle='-', alpha=0.5)
# Форматирование подписей
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m.%y. %H:%M'))
plt.xticks(rotation=45)
plt.title('Сезонная компонента (первые 7 дней данных)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(14,3))
plt.plot(decompose.resid.iloc[:])
plt.grid()
plt.title('Остатки')
plt.xticks(rotation=45)
plt.show()

In [ ]:
decompose.resid.mean(), decompose.resid.std()

In [ ]:

# Параметры
period = 24
all_seasonal = []

plt.figure(figsize=(8, 5))

# Проходим по датчикам, как в ACF
for site, group in df.groupby('site'):
    # Важно: для seasonal_decompose нужен непрерывный ряд без пропусков
    # Берем срез данных (например, первый месяц), ресемплим по часам
    y = group.set_index('time')['pm25'].resample('H').mean().fillna(method='ffill')
    
    if len(y) > period * 2:
        res = seasonal_decompose(y, model='additive', period=period)
        
        # Берем только один цикл (24 часа) из полученной компоненты
        # Seasonal всегда повторяется, так что первых 24 значений достаточно
        seasonal_cycle = res.seasonal.iloc[:period].values
        all_seasonal.append(seasonal_cycle)
        
        # Рисуем "спагетти"
        plt.plot(seasonal_cycle, color='gray', alpha=0.2, linewidth=1)

# Средний ритм города
mean_seasonal = np.mean(all_seasonal, axis=0)
plt.plot(mean_seasonal, color='#1f77b4', linewidth=3, label='Средний суточный ритм сети')

# Оформление в стиле ACF
plt.title('Cезонные компоненты всех узлов мониторинга',fontsize=16)
plt.xlabel('Час суток',fontsize=16)
plt.ylabel('Амплитуда колебаний',fontsize=16)
# plt.axvline(x=6, color='red', linestyle=':', alpha=1, label='Утренний пик')
# plt.axvline(x=20, color='red', linestyle=':', alpha=1, label='Вечерний пик')
plt.xticks(fontsize=16)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
weather = df[df["site"]==3857].drop(['site', 'lat', 'long', 'h', 'pm25'],axis=1).reset_index(drop=True)
hourly_avg = df.groupby('time')['pm25'].median()
mean_data = pd.merge(weather, hourly_avg, on="time")
mean_data = mean_data.set_index('time',drop=True)

In [ ]:
decompose = seasonal_decompose(mean_data['pm25'],period=24*30)
plt.figure(figsize=(8,6))
plt.subplot(3,1,1)
plt.plot(decompose.trend.iloc[:])
plt.xticks(rotation=45)
plt.show()

In [ ]:
for site, group in df.groupby('site'):
    group['time'] = pd.to_datetime(group['time'])
    decompose = seasonal_decompose(group.set_index('time')['pm25'],period=24*30)
    print(site)
    plt.figure(figsize=(8,6))
    plt.subplot(3,1,1)
    plt.plot(decompose.trend.iloc[:])
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
df['hour'] = df['time'].dt.hour
profile = df.groupby(['hour'])[['pm25']].median().reset_index(drop=True)
profile.rename(columns={'pm25':'pm_trend'},inplace=True)
profile['hour'] = [*range(24)]

In [ ]:
df = df.merge(profile, on=['hour'],how='left')

In [ ]:
x = np.arange(24)
y = (0.326)*np.cos(2*np.pi/24*x) + \
    (-0.255)*np.sin(2*2*np.pi/24*x)
target = (profile['pm_trend']-profile['pm_trend'].min())/(profile['pm_trend'].max()-profile['pm_trend'].min())-0.5

plt.plot(target)
plt.plot(y)
plt.show()

In [ ]:
import torch
from kan import KAN

dataset = {
    'train_input': torch.tensor(x, dtype=torch.float32).unsqueeze(1),
    'train_label': torch.tensor(target, dtype=torch.float32).unsqueeze(1),
    'test_input': torch.tensor(x, dtype=torch.float32).unsqueeze(1),
    'test_label': torch.tensor(target, dtype=torch.float32).unsqueeze(1),
}

In [ ]:
trend_model = KAN([1,4,1])

In [ ]:
res = trend_model.fit(dataset,steps=100,lamb=0.001,lr=1)

In [ ]:
res = trend_model.fit(dataset,steps=50,lamb=0,lr=0.1)

In [ ]:
# trend_model = trend_model.prune()
trend_model.plot()

In [ ]:
plt.plot(trend_model(dataset['train_input']).detach().numpy())
plt.plot(target)
plt.show()

In [ ]:
trend_model.auto_symbolic()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

# 1. Восстанавливаем ваши медианные значения по графику (аппроксимация на глаз)
# Вы подставьте сюда точный массив своих медиан!
target_values = np.array([
    0.33,  # 0ч
    0.20,  # 1ч
    0.06,  # 2ч
   -0.02,  # 3ч
   -0.06,  # 4ч
   -0.06,  # 5ч
    0.00,  # 6ч
    0.09,  # 7ч
    0.10,  # 8ч
    0.07,  # 9ч
   -0.08,  # 10ч
   -0.20,  # 11ч
   -0.37,  # 12ч
   -0.45,  # 13ч
   -0.50,  # 14ч
   -0.50,  # 15ч
   -0.43,  # 16ч
   -0.30,  # 17ч
   -0.15,  # 18ч
    0.10,  # 19ч
    0.28,  # 20ч
    0.43,  # 21ч
    0.50,  # 22ч
    0.49   # 23ч
])

hours = np.arange(24)

# 2. Определяем функцию Фурье (две гармоники)
def fourier_func(params, t):
    bias, c1, s1, c2, s2 = params
    w = 2 * np.pi / 24  # основная частота (24ч)
    return bias + c1*np.cos(w*t) + s1*np.sin(w*t) + c2*np.cos(2*w*t) + s2*np.sin(2*w*t)

# 3. Функция ошибки (что минимизируем)
def error_func(params, t, y_true):
    return fourier_func(params, t) - y_true

# 4. Подбор коэффициентов методом наименьших квадратов
initial_guess = [0, 0, 0, 0, 0]
res = least_squares(error_func, initial_guess, args=(hours, target_values))
best_params = res.x

# 5. Генерация синтетической кривой
synthetic_values = fourier_func(best_params, hours)

# 6. Визуализация результата для проверки
plt.figure(figsize=(10, 5))
plt.plot(hours, target_values, 'o-', label='Медиана (Ваш график)', alpha=0.6)
plt.plot(hours, synthetic_values, 'r--', label='Формула (Фурье)', linewidth=2)
plt.title('Аппроксимация суточного тренда')
plt.grid(True)
plt.legend()
plt.xticks(hours)
plt.show()

# 7. Вывод итоговой формулы
b, c1, s1, c2, s2 = best_params
print("Итоговые коэффициенты:")
print(f"Bias: {b:.4f}, C1: {c1:.4f}, S1: {s1:.4f}, C2: {c2:.4f}, S2: {s2:.4f}")

omega = "2*pi/24"
formula = f"SyntTrend(t) = {b:.3f} + ({c1:.3f})*cos({omega}*t) + ({s1:.3f})*sin({omega}*t) + ({c2:.3f})*cos(2*{omega}*t) + ({s2:.3f})*sin(2*{omega}*t)"
print("\nВАША СИНТЕТИЧЕСКАЯ ФОРМУЛА:")
print(formula)

In [ ]:
plt.plot((df[df['site']==3915]['pm25']/df[df['site']==3915]['pm_trend']).values[:1000])
plt.show()

# Проверка на стационарность

In [ ]:
for site, group in df.groupby('site'):
    result = adfuller(group['pm25'], autolag='AIC')
    print(f'{site} pvalue: {result[1]:.4f}')

# Поиск тренда

In [ ]:
# Ядро Епанечникова
def EP_core(x):
    if abs(x) <= 1:
        return 0.75*(1-x**2)
    elif abs(x) > 0:
        return 0

# Гауссово ядро
def GAUSS_core(x):
    if abs(x) <= 1:
        return (1+2*abs(x))*((1-2*abs(x))**2)
    elif abs(x) > 1:
        return 0

# Треугольное ядро
def TR_core(x):
    if abs(x) <= 1:
        return 1-abs(x)
    elif abs(x) > 1:
        return 0

def error(e):
    raise Exception('Incorrect core')
    
cores = {'EP': EP_core,
         'GAUSS': GAUSS_core,
         'TR': TR_core}

# Ядерная регрессия

def restored_x(vector_x, h, X, Y, core):
    denominator = 0
    numerator = 0
    sub = 1
    
    # знаменатель
    for i in range(len(X)):
        for j in range(len(X[i])):
            sub *= core((vector_x[j]-X[i][j])/h)
        denominator += sub
        sub = 1
            
    # числитель
    for i in range(len(X)):
        for j in range(len(X[i])):
            sub *= core((vector_x[j]-X[i][j])/h)
        numerator += sub*Y[i]
        sub = 1
                
    output = numerator/(denominator+0.0000000001)
    
    return output

def core_regression(data_x, data_y, h, core_name):
    core = cores.get(core_name, error)
    restored_regression = []
    i = 1
    for x in data_x:
        restored_regression.append(restored_x(x, h, data_x, data_y, core))
        clear_output(wait=True)
        print(f'{round(i/len(data_x)*100,3)}%')
        i += 1
    # print('Complete')
    return np.array(restored_regression)

In [ ]:
site1

In [ ]:
site1 = df[df['site'] == 3847].reset_index(drop=True)
site1['time'] = pd.to_datetime(site1['time'])
site1 = site1.set_index('time')

window = 24
data_len = site1.shape[0]
edge = (data_len // window) * window

site1 = site1.iloc[:edge]

In [ ]:
WINDOW = 24
trend1 = site1['pm25'].rolling(window=WINDOW, center=True).mean().values.flatten()
trend1[:WINDOW//2] = trend1[WINDOW//2]
trend1[site1.shape[0]-WINDOW//2:] = trend1[site1.shape[0]-WINDOW//2]

In [ ]:
plt.plot((site1['pm25']-trend1)[:1000])
# plt.plot(site1.index, trend)
plt.xticks(rotation=45)
plt.show()

In [ ]:
dec = seasonal_decompose(site1['pm25']-trend1,period=24)

In [ ]:
plt.plot(site1['pm25'].iloc[:1000])
plt.plot((dec.seasonal+trend1)[:1000])
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.plot(dec.seasonal[:100])
plt.xticks(rotation=45)
plt.show()

# Сингулярное разложение

In [ ]:
def components(U,S,VT,a,b):
    u = U[:, range(a, b)]
    s = np.diag(S)[a:b, a:b]
    vt = VT[range(a, b), :]
    res_a = u @ s @ vt
    return res_a

In [ ]:
for site, group in df.groupby('site'):
    print(site, group.shape[0])

In [ ]:
reshaped_data = site1['inflow'].values.reshape(data_len//window,window)
u, s, vt = svd(reshaped_data)

In [ ]:
plt.plot([*range(len(s))],s,'b-o')
plt.xlabel('Сингулярные числа')
plt.ylabel('Значения сингулярных чисел')
plt.show()

In [ ]:
harmonic1 = components(u,s,vt,0,1).flatten()
harmonic2 = components(u,s,vt,1,2).flatten()
harmonic3 = components(u,s,vt,2,3).flatten()
harmonic4 = components(u,s,vt,3,4).flatten()
harmonic4 = components(u,s,vt,4,5).flatten()

In [ ]:
plt.figure(figsize=(12,8))
for i, harm in enumerate([harmonic1,
                          harmonic2,
                          harmonic3,
                          harmonic4]):
    plt.subplot(2,2,i+1)
    plt.plot(pd.to_datetime(site1.index), harm, label=f'harmonic {i+1}')
    plt.xticks(rotation=45)
    plt.legend()
plt.show()

In [ ]:
np.sqrt(mean_squared_error(site1['pm25'],harmonic1+harmonic2+harmonic3+harmonic4))

In [ ]:
mean_absolute_error(site1['pm25'],harmonic1)

In [ ]:
sns.heatmap(np.corrcoef([harmonic1,harmonic2,harmonic3,harmonic4]).round(2),annot=True)
plt.show()

In [ ]:
sns.heatmap(np.corrcoef([harmonic1,np.abs(harmonic2+harmonic3+harmonic4)]).round(2),annot=True)
plt.show()

In [ ]:
seasonal1 = harmonic1-trend1+harmonic2+harmonic3+harmonic4

In [ ]:
sns.heatmap(np.corrcoef([harmonic1,seasonal1]).round(2),annot=True)
plt.show()

In [ ]:
plt.plot(site1.index[:1000],trend1[:1000],alpha=0.7,color='black')
plt.plot(site1.index[:1000],harmonic1[:1000],alpha=0.7)
plt.plot(site1.index[:1000],(harmonic1-trend1+harmonic2+harmonic3+harmonic4)[:1000],alpha=0.7)
plt.xticks(rotation=45)
plt.show()

In [ ]:
u1, s1, vt1 = svd((site1['pm25'].values-trend1).reshape(data_len//window,window))
harmonic11 = components(u1,s1,vt1,0,1).flatten()
harmonic12 = components(u1,s1,vt1,1,2).flatten()
harmonic13 = components(u1,s1,vt1,2,3).flatten()
harmonic14 = components(u1,s1,vt1,3,4).flatten()

In [ ]:
plt.plot([*range(len(s1))],s1,'b-o')
plt.xlabel('Сингулярные числа')
plt.ylabel('Значения сингулярных чисел')
plt.show()

In [ ]:
plt.figure(figsize=(12,8))
for i, harm in enumerate([harmonic11,
                          harmonic12,
                          harmonic13,
                          harmonic14]):
    plt.subplot(2,2,i+1)
    plt.plot(pd.to_datetime(site1.index), harm, label=f'harmonic {i+1}')
    plt.legend()
plt.title('SVD разложение гармоник')
plt.show()

In [ ]:
seasonal2 = harmonic11

In [ ]:
sns.heatmap(np.corrcoef([trend1,harmonic11,harmonic1]).round(2),annot=True)
plt.show()

In [ ]:
plt.plot((trend1+harmonic11)[:20000])
plt.plot(site1['pm25'].values[:20000],alpha=0.5)

In [ ]:
plt.plot(site1['pm25'].values[:1000],alpha=0.5)
plt.plot(trend1[:1000])
plt.plot(trend2[:1000])

# Метод "Гусеница"

In [ ]:
def get_ssa_components(series, window=24, components_range=(0, 1)):
    """
    Выполняет SSA декомпозицию и возвращает восстановленный ряд 
    для заданного диапазона сингулярных чисел.
    """
    N = len(series)
    L = window
    K = N - L + 1
    
    # 1. Вложение (Embedding) - создание траекторной матрицы (Ганкелевой)
    # Используем stride_tricks для экономии памяти при создании матрицы
    from numpy.lib.stride_tricks import as_strided
    from scipy.sparse.linalg import svds


    X = as_strided(series, shape=(K, L), strides=(series.strides[0], series.strides[0]))
    
    # 2. Разложение (SVD)
    # Вместо: U, S, VT = svd(X, full_matrices=False)
    U, S, VT = svds(X, k=window-1)

    # plt.plot([*range(len(S))][::-1],S,'b-o')
    # plt.xlabel('Сингулярные числа')
    # plt.ylabel('Значения сингулярных чисел')
    # plt.show()
    
    # ВАЖНО: svds возвращает S в порядке ВОЗРАСТАНИЯ. 
    # Нужно развернуть массивы для привычной логики:
    S = S[::-1]
    U = U[:, ::-1]
    VT = VT[::-1, :]
    
    # 3. Группировка и восстановление аппроксимированной матрицы
    a, b = components_range
    # Вычисляем матрицу только для нужных компонент
    X_hat = (U[:, a:b] * S[a:b]) @ VT[a:b, :]
    
    # 4. Диагональное усреднение (Anti-diagonal averaging)
    # Превращаем матрицу K x L обратно в ряд длиной N
    recovered_series = np.zeros(N)
    counts = np.zeros(N)
    
    for i in range(K):
        for j in range(L):
            recovered_series[i + j] += X_hat[i, j]
            counts[i + j] += 1
            
    return recovered_series / counts

In [ ]:
# Пример использования для выделения первой компоненты (Тренда)
trend2 = get_ssa_components(site1['pm25'].values, window=48, components_range=(0, 3))

# Пример для выделения сезонности (обычно компоненты 1 и 2 или 1-3)
seasonal3 = get_ssa_components(site1['pm25'].values, window=48, components_range=(3, 9))

In [ ]:
plt.plot(trend2)

In [ ]:
plt.plot(seasonal3)

# FFT

In [ ]:
def fourierplot(t, data):
    fourier = np.fft.fft(data)
    freq = np.fft.fftfreq(t.shape[0])
    magnitude = np.abs(fourier)

    plt.figure(figsize=(8,3))
    plt.plot(freq[:t.shape[0]//6], magnitude[:t.shape[0]//6],color="orange",linewidth=3,alpha=1)#,markerfmt='.',linefmt='c-')
    plt.title('Фурье анализ частот', fontsize=12)
    plt.axvline(1/24,color='red',label='Суточный период',alpha=0.65,linestyle='--',linewidth=2)
    plt.axvline(1/12,color='blue',label='12-часовой период',alpha=0.65,linestyle='--',linewidth=2)
    plt.xlabel('Частота [1/час]', fontsize=10)
    plt.ylabel('Амплитуда', fontsize=10)
    plt.legend()
    ax = plt.gca()  
    # ax.set_facecolor("black")
    plt.grid()
    plt.plot()

In [ ]:
fourierplot(site1.index, trend1)

In [ ]:
fourierplot(site1.index, trend2)

In [ ]:
fourierplot(site1.index, seasonal1)

In [ ]:
fourierplot(site1.index, seasonal2)

In [ ]:
fourierplot(site1.index, seasonal3)

In [ ]:
fourierplot(site1.index, site1['pm25']-trend1)

In [ ]:
fourierplot(site1.index, site1['pm25']-trend2)

In [ ]:
fourierplot(site1.index, site1['pm25']-trend2)

In [ ]:
site1

In [ ]:
np.corrcoef([seasonal_decompose(site1['pm25'],period=24).seasonal.values, seasonal3])

In [ ]:
N1, N2 = 0, 24*7

plt.plot(trend2[N1:N2], label='trend')
plt.plot((site1['pm25']-trend2).values[N1:N2], label='high frequency harmonic')
plt.plot((site1['pm25']).values[N1:N2], label='original')
plt.xticks(rotation=45)
plt.legend()
plt.show()

In [ ]:
site1['hfh'] = site1['pm25']-trend2
site1['trend'] = trend2

In [ ]:
plt.figure(figsize=(25,16))
sns.heatmap(site1.corr().round(2), annot=True)
plt.show()

In [ ]:
plt.figure(figsize=(25,16))
sns.heatmap(site1[site1['hfh'] < 0].corr().round(2), annot=True)
plt.show()

In [ ]:
plt.figure(figsize=(25,16))
sns.heatmap(site1[site1['hfh'] >= 0].corr().round(2), annot=True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import as_strided
from scipy.sparse.linalg import svds

def plot_ensemble_analysis(df, window=48, components_range=(3, 11)):
    """
    Строит ансамбль сингулярных чисел и Фурье-спектров для всех датчиков.
    """
    sites = df['site'].unique()
    
    # Списки для хранения результатов
    all_singular_values = []
    all_fourier_mags = []
    freqs = None
    
    # 1. Сбор данных по всем датчикам
    for site in sites:
        series = df[df['site'] == site]['pm25'].resample('H').mean().fillna(method='ffill').values
        if len(series) < window * 2: continue
            
        # SSA часть (только разложение для получения S)
        N, L = len(series), window
        K = N - L + 1
        X = as_strided(series, shape=(K, L), strides=(series.strides[0], series.strides[0]))
        _, S, _ = svds(X, k=window-1)
        S = S[::-1]
        all_singular_values.append(S / S.max()) # Нормируем для визуального сравнения
        
        # SSA восстановление для Фурье (берём компоненты 3-10)
        # Здесь используем твою логику восстановления
        recovered = get_ssa_components(series, window=window, components_range=components_range)
        
        # Фурье часть
        fourier = np.fft.fft(recovered)
        current_freqs = np.fft.fftfreq(len(recovered))
        magnitude = np.abs(fourier)
        
        # Нормируем амплитуду (от 0 до 1)
        magnitude = magnitude / magnitude.max()
        
        all_fourier_mags.append(magnitude)
        freqs = current_freqs

    # # 2. Визуализация
    # fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10))

    # # График сингулярных чисел
    # for s_vals in all_singular_values:
    #     ax1.plot(s_vals, color='gray', alpha=0.2, linewidth=1)
    # ax1.plot(np.median(all_singular_values, axis=0), color='blue', linewidth=3, label='Медиана')
    # ax1.set_title('Нормированные сингулярные числа (все датчики)',fontsize=12)
    # ax1.set_xlabel('Индекс числа',fontsize=16)
    # ax1.set_ylabel('Относительная значимость',fontsize=16)
    # ax1.grid(True, linestyle=':')

    # График Фурье-ансамбля

    plt.figure(figsize=(12,5))
    
    limit = len(freqs) // 6 # Ограничение частоты как в твоей функции
    for mag in all_fourier_mags:
        plt.plot(freqs[:limit], mag[:limit], color='orange', alpha=0.15, linewidth=1)
    
    mean_mag = np.mean(all_fourier_mags, axis=0)
    plt.plot(freqs[:limit], mean_mag[:limit], color='red', linewidth=2, label='Средний спектр сети')
    
    plt.axvline(1/24, color='black', linestyle='--', alpha=0.8, label='24 часа')
    plt.axvline(1/12, color='black', linestyle='-.', alpha=0.8, label='12 часов')
    plt.title(f'Частотный анализ сети после SSA, компоненты [{components_range[0]}-{components_range[1]})',fontsize=16)
    plt.xlabel('Частота [1/час]',fontsize=16)
    plt.ylabel('Нормированная амплитуда',fontsize=16)
    plt.xticks([1/168, 1/48, 1/24, 1/16, 1/12, 1/8], ['1/168', '1/48', '1/24', '1/16', '1/12', '1/8'],fontsize=16)
    plt.legend(fontsize=16)
    plt.grid(True, linestyle=':')

    # plt.tight_layout()
    plt.show()
    
    return recovered


In [ ]:
df = df.set_index(['time'])

In [ ]:
recovered = plot_ensemble_analysis(df,48,(1,2))

In [ ]:
s = df[df['site']==3857]
s['1_3_harm'] = recovered

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(s['1_3_harm'].values)

plt.subplot(1,2,2)
plt.plot(s['1_3_harm'].values[:2000])

plt.show()

In [ ]:
plt.figure(figsize=(25,20))
sns.heatmap(s.corr().round(2),annot=True)
plt.show()

In [ ]:
plot_ensemble_analysis(df,48,(1,12))

In [ ]:
recovered = plot_ensemble_analysis(df,48,(0,3))

In [ ]:
recovered = plot_ensemble_analysis(df,48,(3,9))

# Связь с целевой переменной

In [ ]:
sns.heatmap(np.corrcoef([seasonal1,seasonal2,seasonal3,site1['pm25'].values]).round(2),annot=True)
plt.show()

In [ ]:
site2 = df[df['site']==3854][:edge]

In [ ]:
sns.heatmap(np.corrcoef([site1['pm25'].values,site2['pm25'].values]).round(2),annot=True)
plt.show()

In [ ]:
k = 18
plt.plot(site1['pm25'].values[(k-1)*120:k*150])
plt.plot(1000000*site1['inflow'].values[(k-1)*120:k*150])
plt.show()

# Автоматизация обработки данных для всех датчиков и их признаков

In [ ]:
cycle_features = ['pm25','T','Po','U']

site_trends = []
site_seasonals = []

trends = []
seasonals = []

sites_group = df.groupby('site')
for f in cycle_features:
    clear_output(wait=True)
    print('Признак', f)
    # Анализ автокорреляции
    for site, group in sites_group:
        clear_output(wait=True)
        print(f'----{site}----')
        f_data = group[f]
        plot_acf(f_data, alpha=0.05, lags=24*3)
        plot_pacf(f_data, alpha=0.05, lags=24*3)
        plt.show()

        if f == 'pm25':
            window = 24
        else:
            window = int(input('Размер окна: '))
        # Разделение на тренд и сезонность
        print('Тренд')
        trend = get_ssa_components(f_data.values, window=window, components_range=(0, 1))
        print('Сезонная компонента')
        seasonal = get_ssa_components(f_data.values, window=window, components_range=(1, 5))
        plt.figure(figsize=(12,12))
        plt.subplot(2,2,1)
        plt.plot(f_data.values,alpha=0.5,label='orig')
        plt.plot(trend,label='svd trend')
        plt.legend()
        plt.subplot(2,2,2)
        plt.plot(f_data.values[:1000],alpha=0.5,label='orig')
        plt.plot(trend[:1000],label='svd trend')
        plt.legend()
        plt.subplot(2,2,3)
        plt.plot(f_data.values,alpha=0.5,label='orig')
        plt.plot(seasonal,label='seasonal')
        plt.legend()
        plt.subplot(2,2,4)
        plt.plot(f_data.values[:1000],alpha=0.5,label='orig')
        plt.plot(seasonal[:1000],label='seasonal')
        plt.legend()
        plt.show()
    
        fourierplot(f_data.index, trend)
        plt.show()
        fourierplot(f_data.index, seasonal)
        plt.show()

        if f == 'pm25':
            wait = input('Перейти к слудующему датчику?')
            site_trends.append(trend)
            site_seasonals.append(seasonal)
        else:
            wait=input()
            trends.append(trend)
            seasonals.append(seasonal)
            break
    
        

In [ ]:
plt.plot(trends[0])
plt.show()
plt.plot(trends[1])
plt.show()
plt.plot(trends[2])
plt.show()

In [ ]:
plt.plot(seasonals[0])
plt.show()
plt.plot(seasonals[1])
plt.show()
plt.plot(seasonals[2])
plt.show()

# Проверка модуля

In [ ]:
class SSAPreprocessor:
    def __init__(self):
        self.window = None
        self.mean = None
        self.std = None
        
        self.U = None
        self.S = None
        self.Vt = None
        
        self.N = None
        self.L = None
        self.K = None
        
    def fit(self, series, window):
        self.window = window
        self.N = len(series)
        self.L = self.window
        self.K = self.N - self.L + 1
        
        self.mean = np.mean(series)
        self.std = np.std(series)
        data_std = ((series - self.mean) / self.std).values
        
        X = as_strided(data_std, shape=(self.K, self.L),
                       strides=(data_std.strides[0],
                                data_std.strides[0]))
        U, S, VT = svds(X, k=self.window-1)
        self.S = S[::-1]
        self.U = U[:, ::-1]
        self.VT = VT[::-1, :]

    def decompose(self, components_range=(0,5)):
        a, b = components_range
        X_hat = (self.U[:, a:b] * self.S[a:b]) @ self.VT[a:b, :]
        recovered_series = np.zeros(self.N)
        counts = np.zeros(self.N)
        for i in range(self.K):
            for j in range(self.L):
                recovered_series[i + j] += X_hat[i, j]
                counts[i + j] += 1
        return recovered_series / counts
        
    def transform(self, series):
        data_std = (series - self.mean) / self.std
        return data_std

    def inverse_transform(self, series):
        data = (series * self.std) + self.mean
        return data

In [ ]:
pm_prep = SSAPreprocessor()
T_prep = SSAPreprocessor()
U_prep = SSAPreprocessor()
P_prep = SSAPreprocessor()

In [ ]:
site1 = df[df['site'] == 4227].reset_index(drop=True).iloc[:edge]
site1['time'] = pd.to_datetime(site1['time'])
site1 = site1.set_index('time')

In [ ]:
pm_prep.fit(site1['pm25'], 24)
T_prep.fit(site1['T'], 24)
U_prep.fit(site1['U'], 24)
P_prep.fit(site1['Po'], 48)

In [ ]:
pm_trend = pm_prep.decompose(components_range=(0,1))
pm_seas = pm_prep.decompose(components_range=(1,5))

T_trend = T_prep.decompose(components_range=(0,1))
T_seas = T_prep.decompose(components_range=(1,5))
U_trend = U_prep.decompose(components_range=(0,1))
U_seas = U_prep.decompose(components_range=(1,5))
P_trend = P_prep.decompose(components_range=(0,1))
P_seas = P_prep.decompose(components_range=(1,5))

In [ ]:
plt.plot(site1['pm25'].values[0 *24*30*12 : 1 *24*30*12])
plt.plot(site1['pm25'].values[1 *24*30*12 : 2 *24*30*12])
plt.plot(site1['pm25'].values[2 *24*30*12 : 3 *24*30*12])
plt.plot(site1['pm25'].values[3 *24*30*12 : 4 *24*30*12])
plt.plot(site1['pm25'].values[4 *24*30*12 : 5 *24*30*12])
plt.show()

In [ ]:
f = 'pm25'
five_years = np.array([site1[f].values[0 *24*30*12 : 1 *24*30*12],
                       site1[f].values[1 *24*30*12 : 2 *24*30*12],
                       site1[f].values[2 *24*30*12 : 3 *24*30*12],
                       site1[f].values[3 *24*30*12 : 4 *24*30*12],
                       site1[f].values[4 *24*30*12 : 5 *24*30*12]])

mean_f = np.median(five_years,axis=0)
std_f = np.std(five_years,axis=0)
plt.plot(mean_f,color='blue')
plt.fill_between([*range(mean_f.shape[0])],
                 mean_f-std_f,
                 mean_f+std_f,
                 alpha=0.5,
                 color='blue')
plt.show()

In [ ]:
[site1.index[:mean_f.shape[0]]]

In [ ]:
mean_f = pd.DataFrame(zip(site1.index[:mean_f.shape[0]],mean_f), columns=['time','pm25'])
mean_f['time'] = pd.to_datetime(mean_f['time'])
mean_f = mean_f.set_index('time',drop=True)

In [ ]:
WINDOW = 23*30
f_trend = mean_f['pm25'].rolling(window=WINDOW, center=True).mean().values.flatten()
f_trend[:WINDOW//2] = f_trend[WINDOW//2]
f_trend[f_trend.shape[0]-WINDOW//2:] = f_trend[f_trend.shape[0]-WINDOW//2]

In [ ]:
trend_nonpreg = core_regression([[x] for x in range(mean_f.shape[0])], mean_f['pm25'].values, 500, 'EP')

In [ ]:
plt.plot(mean_f.values,alpha=0.5)
plt.plot(f_trend)
plt.plot(trend_nonpreg)

In [ ]:
non_trend = mean_f['pm25'] / trend_nonpreg

In [ ]:
pm_prep = SSAPreprocessor()
pm_prep.fit(non_trend, 24)

sub_trend = pm_prep.decompose((0,1))
sub_seas = pm_prep.decompose((1,5))

In [ ]:
plt.plot(sub_trend)
plt.show()

In [ ]:
plt.plot([*range(0,1*24)],sub_seas[1*24:24*2])
plt.plot([*range(1*24,2*24)],sub_seas[2*24:24*3])
plt.plot([*range(2*24,3*24)],sub_seas[3*24:24*4])
plt.plot([*range(3*24,4*24)],sub_seas[4*24:24*5])
plt.plot([*range(4*24,5*24)],sub_seas[5*24:24*6])
plt.plot([*range(5*24,6*24)],sub_seas[6*24:24*7])
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_phase_evolution(df, column='pm25'):
    plt.figure(figsize=(8, 4))
    
    # Создаем лаг t+1
    df_plot = df.copy()
    df_plot['next_val'] = df_plot[column].shift(-1)
    df_plot['month'] = df_plot.index.month
    
    # Убираем NaN после shift
    df_plot = df_plot.dropna()
    
    # Строим фазовый портрет с цветовой кодировкой по месяцам
    scatter = plt.scatter(df_plot[column], df_plot['next_val'], 
                         c=df_plot['month'], cmap='coolwarm', 
                         alpha=0.6, s=5)
    
    # Линия идеального совпадения (наивный прогноз)
    max_val = df_plot[column].max()
    plt.plot([0, max_val], [0, max_val], 'k--', alpha=0.5, label='y = x (Naive)')
    
    plt.colorbar(scatter, label='Месяц (1=Янв, 12=Дек)')
    plt.title(f'Эволюция динамики {column}: Связь t и t+1 по сезонам')
    plt.xlabel('Текущая концентрация $PM_{2.5} (t)$')
    plt.ylabel('Концентрация через час $PM_{2.5} (t+1)$')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

plot_phase_evolution(site1)

In [ ]:
fourierplot(np.arange(sub_trend.shape[0]), sub_seas)

In [ ]:
diff = pd.DataFrame(a,columns=['diff'])
diff['time'] = pd.to_datetime(site1.index[:len(a)])
diff = diff.set_index('time', drop=True)

In [ ]:
dec = seasonal_decompose(diff)

In [ ]:
plt.plot(dec.seasonal[:100])

In [ ]:
fourierplot(diff.index, diff['diff'])

In [ ]:
shift1 = np.array([0,*five_years[1][:-1]])
diff1 = (five_years[1] - shift1)[1:]
shift2 = np.array([0,*mean_f[:-1]])
diff2 = (mean_f - shift2)[1:]

In [ ]:
np.corrcoef([diff1,diff2])

In [ ]:
plt.plot(diff1)
plt.plot(diff2)
plt.show()

In [ ]:
plt.plot(five_years[0]-five_years[1])
plt.plot(five_years[1]-five_years[2])
plt.plot(five_years[2]-five_years[3])
plt.plot(five_years[3]-five_years[4])
plt.show()

In [ ]:
diff = pd.DataFrame(np.clip(pm_trend_prep.transform(site1['pm25']).values[:]/pm_trend[:],-4,4),columns=['diff'])
diff['time'] = pd.to_datetime(site1.index[:site1.shape[0]])
diff = diff.set_index('time', drop=True)

In [ ]:
dec = seasonal_decompose(diff)

In [ ]:
plt.plot(dec.seasonal[:100])
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.plot(pm_trend_prep.transform(site1['pm25']).values[:2000],alpha=0.5)
plt.plot(pm_seas[:2000])
plt.plot(np.clip(pm_trend_prep.transform(site1['pm25']).values[:2000]/pm_trend[:2000],-4,4))

In [ ]:
t = pd.DataFrame([*zip(site1.index,
                       pm_trend,
                       pm_seas,
                       T_trend,
                       U_trend,
                       P_trend,
                       T_seas,
                       U_seas,
                       P_seas)],
                 columns=['time','pm25t','pm25s','Tt','Ut','Pt','Ts','Us','Ps'])

In [ ]:
diff = (t['pm25t'] - t['pm25t'].shift(1)).iloc[1:]
t = t.drop(index=0)
t['diff'] = diff

In [ ]:
t

In [ ]:
sns.heatmap(t.corr().round(2),annot=True)
plt.show()

In [ ]:
plt.plot(t['diff'].values[:1000],alpha=0.5)
plt.plot(t['pm25t'].values[:1000])
plt.show()

# Вейвлет-разложение

In [ ]:
import pywt
s = df[df['site']==3857]
n = (s.shape[0] // (2**3)) * (2**3)

coeffs = pywt.swt(s['pm25'].values[:n], 'db4', level=3,trim_approx=True)
titles = ['A3 (Trend)', 'D3 (Daily)', 'D2 (Meso)', 'D1 (Fast)']

plt.figure(figsize=(10.2,4))
plt.plot(s['pm25'].values[:1000])
plt.show()
plt.figure(figsize=(10,14))
for i, c in enumerate(coeffs):
    plt.subplot(4, 1, i+1)
    plt.plot(c[:1000])
    # plt.title(titles[i])
plt.show()

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
r2_score(s['pm25'].values[:n], coeffs[0])

In [ ]:
dec = seasonal_decompose(pd.DataFrame(zip(coeffs[3],s['time'].values[:n]),columns=['pm25','time']).set_index('time') )

In [ ]:
plt.plot(dec.seasonal[:100])
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.plot(coeffs[0][:240])
plt.plot(coeffs[1][:240])
plt.plot(coeffs[2][:240])
plt.plot(coeffs[3][:240])
plt.show()

# Датасет от Kaggle

In [ ]:
kdf = pd.read_csv('data/GlobalWeatherRepository.csv')

In [ ]:
kdf

In [ ]:
kdf.info()

In [ ]:
plt.figure(figsize=(25,15))
sns.heatmap(kdf.select_dtypes(include=['number']).corr().round(2),annot=True)
plt.show()

In [ ]:
plt.scatter(kdf['uv_index'], kdf['air_quality_PM2.5'])
plt.show()

# Кросс-корреялционный анализ

In [ ]:
sensors = pd.read_csv('data/min_sensors_project_9.csv')
sensors

In [ ]:
season.values

In [ ]:
df = df.set_index('time')
df['time'] = df.index

In [ ]:
# season = pd.date_range(
#     "11.01.2021",
#     "03.01.2022",
#     freq='1h'
# )
# season = pd.date_range(
#     "06.01.2021",
#     "10.01.2022",
#     freq='1h'
# )

# Добавление лагов

# df = df.sort_values(['site', 'time'])

df['pm25_lag1'] = df.groupby('site')['pm25'].shift(1)

df = df.dropna(subset=['pm25_lag1'])

df['diff'] = df['pm25'] - df['pm25_lag1']


In [ ]:
df.columns

In [ ]:
pm25_wide = df.pivot(index='time', columns='site', values='diff')

In [ ]:
corr_tensor = []
sites = [sensors[sensors['id']==int(site)]['name'].values[0] for site in pm25_wide.columns]
for t in range(0,25):
    corr_matrix = []
    for site in pm25_wide.columns:
        site_col = pm25_wide[int(site)]
        shift_df = pm25_wide.shift(t)
        shift_df = shift_df.dropna()
        shift_df[int(site)] = site_col.values[t:]
        corr_matrix.append(shift_df.corr().loc[int(site)].values)
        
    corr_tensor.append(corr_matrix)


In [ ]:
corr_df = pd.DataFrame(corr_tensor[24],columns=sites,index=sites)

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(corr_df, annot=True)
plt.show()

In [ ]:
colors = cm.plasma(np.linspace(0, 1, 25))[::-1]
for i, site in enumerate(sites):
    plt.figure(figsize=(16,4))
    for t in range(25):
        plt.plot(sites, corr_tensor[t][i], 'o-', label=f't={t}',color=colors[t])
    plt.title(site)
    plt.xticks(rotation=45)
    plt.ylabel('Корреляция')
    plt.ylim(-0.2,0.6)
    plt.show()

In [ ]:
def calculate_wind_lagged_corr(df, target_site, wind_condition_func, max_lag=24):
    """
    Вычисляет кросс-корреляцию первых разностей ('diff') между целевым датчиком 
    (в момент t - lag) и остальными датчиками (в момент t) при определенном ветре.
    """
    # Шаг 1. Переводим датасет из длинного формата в широкий по столбцу 'diff'
    # Каждая станция станет отдельным столбцом
    pivot_diff = df.pivot(index='time', columns='site', values='diff')
    
    # Шаг 2. Заполняем пропуски во времени, создавая непрерывную почасовую сетку.
    # Это ГЛАВНОЕ: теперь сдвиг на 1 позицию ВСЕГДА равен 1 часу!
    all_hours = pd.date_range(start=pivot_diff.index.min(), end=pivot_diff.index.max(), freq='h')
    pivot_diff = pivot_diff.reindex(all_hours)
    
    # Извлекаем метеоданные для каждого часа (привязываемся к целевой станции)
    meteo = df[df['site'] == target_site].set_index('time')[['DD', 'Ff']]
    meteo = meteo.reindex(all_hours)
    
    results = {}
    other_sites = [col for col in pivot_diff.columns if col != target_site]
    
    # Шаг 3. Итерируемся по лагам
    for lag in range(max_lag + 1):
        # Сдвигаем целевой датчик на 'lag' часов НАЗАД. 
        # Благодаря непрерывному индексу временной лаг не собьется!
        target_shifted = pivot_diff[target_site].shift(lag)
        
        # Собираем временный датафрейм для сопоставления в момент времени 't'
        temp_df = pd.DataFrame({
            'target_lagged': target_shifted, # значение целевой станции в t - lag
            'DD': meteo['DD'],               # направление ветра в момент t
            'Ff': meteo['Ff']                # скорость ветра в момент t
        })
        
        # Добавляем значения остальных станций в текущий момент 't'
        temp_df = temp_df.join(pivot_diff[other_sites])
        
        # Шаг 4. Применяем фильтрацию по ветру в текущий момент времени 't'
        # (Так как ветер несет массу именно сейчас)
        valid_rows = wind_condition_func(temp_df)
        filtered_df = temp_df[valid_rows]
        
        # Считаем корреляцию между целевым (t-lag) и остальными (t)
        lag_corrs = {}
        for site in other_sites:
            # Метод corr() в pandas автоматически игнорирует пропуски (NaN)
            corr_val = filtered_df['target_lagged'].corr(filtered_df[site])
            lag_corrs[site] = corr_val
            
        results[lag] = lag_corrs
        
    # Возвращаем датафрейм: строки - лаги (0-24), столбцы - другие датчики
    return pd.DataFrame(results).T

# --- ПРИМЕР ИСПОЛЬЗОВАНИЯ ---

# Обязательно переведите столбец времени в datetime перед вызовом функции!
df['time'] = pd.to_datetime(df['time'])


In [ ]:
def specific_wind_condition(data):
    # condition = (data['Ff'] == 0.0)
    condition = (data['DD'] >= 180) & (data['DD'] <= 360) & (data['Ff'] == 3) #& (data['Ff'] <= 3)
    return condition

corr_matrix = calculate_wind_lagged_corr(df, target_site=3857, wind_condition_func=specific_wind_condition)
plt.figure(figsize=(16,4))
sns.lineplot(data=corr_matrix)
plt.legend(loc=1)
plt.xticks([*range(25)])
plt.ylabel('Корреляция')
plt.xlabel('Временной лаг')
plt.show()

In [ ]:
def deg2rad(deg):
    return deg * (np.pi / 180)

def get_distance_m(lat1, lon1, lat2, lon2):
    R = 6371  # Radius of the earth in km
    d_lat = deg2rad(lat2 - lat1)
    d_lon = deg2rad(lon2 - lon1)
    
    a = (np.sin(d_lat / 2) ** 2 +
         np.cos(deg2rad(lat1)) * np.cos(deg2rad(lat2)) *
         np.sin(d_lon / 2) ** 2)
    
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    d = R * c  # Distance in km
    
    return d * 1000


In [ ]:
dist_matrix = [[0 for _ in range(17)] for __ in range(17)]

In [ ]:
for i, site1 in enumerate(df['site'].unique()):
    s1 = df[df['site']==site1]
    lat1 = s1['lat'].values[0]
    long1 = s1['long'].values[0]
    for j, site2 in enumerate(df['site'].unique()):
        s2 = df[df['site']==site2]
        lat2 = s2['lat'].values[0]
        long2 = s2['long'].values[0]

        dist_matrix[i][j] = get_distance_m(lat1,long1,lat2,long2) / 1000

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(dist_matrix, annot=True)
plt.show()

In [ ]:
np.mean(dist_matrix)

In [ ]:
df

In [ ]:
def normalize(data):
    std = np.std(data)
    mean = np.mean(data)
    return (data-mean)/std

In [ ]:
s = df[df['site']==3857]
v = 24*30
plt.figure(figsize=(12,4))
plt.plot(s['time'].values[v*6:v*8],normalize(s['T'].values)[v*6:v*8],label='T')
plt.plot(s['time'].values[v*6:v*8],normalize(s['U'].values)[v*6:v*8],label='U')
plt.plot(s['time'].values[v*6:v*8],normalize(s['Ff'].values)[v*6:v*8],label='Ff')
plt.plot(s['time'].values[v*6:v*8],normalize(s['Po'].values)[v*6:v*8],label='P')
plt.plot(s['time'].values[v*6:v*8],normalize(s['pm25'].values)[v*6:v*8],label='pm25',alpha=0.5)
plt.legend()
plt.xticks(rotation=45)
plt.show()

In [ ]:
v = 24*30*2
plt.figure(figsize=(12,4))
plt.plot(s['time'].values[:v],normalize(s['T'].values)[:v],label='T')
plt.plot(s['time'].values[:v],normalize(s['U'].values)[:v],label='U')
plt.plot(s['time'].values[:v],normalize(s['Ff'].values)[:v],label='Ff')
plt.plot(s['time'].values[:v],normalize(s['Po'].values)[:v],label='P')
plt.plot(s['time'].values[:v],normalize(s['pm25'].values)[:v],label='pm25',alpha=0.5)
plt.legend()
plt.xticks(rotation=45)
plt.show()